# Hi-cGAN Diffusion Model — Google Colab Training & Prediction Notebook
This notebook sets up the environment, trains the Diffusion model where **1 fixed random Gaussian matrix per chromosome** (`data/random_input/chr*.npy`) is used as input and **250kb RNA interaction matrices (`data/rna_gt/chr*.npz`)** are used as Ground Truth targets, and evaluates predictions with `predict.py`.

## 1. Check GPU & Mount Google Drive

In [ ]:
!nvidia-smi

from google.colab import drive
drive.mount('/content/drive')

## 2. Navigate to Project Directory & Install Dependencies
Update `PROJECT_DIR` below to match the folder path in your Google Drive (e.g. `/content/drive/MyDrive/Hi-cGAN`).

In [ ]:
import os

# Path to the Hi-cGAN repository in Google Drive
PROJECT_DIR = '/content/drive/MyDrive/Hi-cGAN'
os.chdir(PROJECT_DIR)
print('Current working directory:', os.getcwd())

!pip install -q cooler h5py scipy matplotlib tqdm scikit-learn

## 3. Clean Old TFRecords (Optional)
Remove any old TFRecord files in `output_model` so new records are created using the per-chromosome fixed random matrices.

In [ ]:
!rm -f hicgan/output_model/*.tfrecord hicgan/output_model/hicgan_metadata.pkl

## 4. Run Diffusion Model Training (`training.py`)

In [ ]:
!python hicgan/training.py \
    --dataDir hicgan/data \
    --splitFile hicgan/data/splits.txt \
    --windowSize 64 \
    --outputFolder hicgan/output_model \
    --epochs 50 \
    --batchSize 16 \
    --timesteps 1000 \
    --learningRatedenoiser 2e-5 \
    --plotFrequency 5 \
    --keepTFRecords

## 5. Predict & Visualize Test Chromosomes (`predict.py`)
Evaluate the trained model on the `test` chromosomes from `splits.txt` (or set `--split train` to evaluate on the exact same training chromosomes/random matrices).

In [ ]:
!python hicgan/predict.py \
    --trainedModel hicgan/output_model/denoiser_final.keras \
    --dataDir hicgan/data \
    --splitFile hicgan/data/splits.txt \
    --split test \
    --windowSize 64 \
    --batchSize 16 \
    --outputFolder hicgan/test_results

## 6. Display Training Loss and Visual Prediction Results

In [ ]:
import glob
from IPython.display import Image, display

loss_plot = 'hicgan/output_model/lossOverEpochs.png'
if os.path.exists(loss_plot):
    print('Loss Over Epochs:')
    display(Image(filename=loss_plot))

for fig_path in sorted(glob.glob('hicgan/test_results/pred_chr*.png')):
    print(fig_path)
    display(Image(filename=fig_path))